# Agents Demo: Modeling, Evaluation, and OpenAI Review

This notebook shows how to orchestrate a full Model Zoo workflow using the `ModelingAgent`, `EvaluationAgent`, and the optional `OpenAIReviewerAgent`.

This walkthrough assumes the Model Zoo already contains the `logit_marketing_v1` configuration and metrics; the agents run directly and rely on the stored metadata.

## Prerequisites
- A Model Zoo API server running locally at `http://localhost:8000`.
- Required packages installed via `pip install -e .` (from the repo root).
- Optional: set `OPENAI_API_KEY` (or pass `api_key` explicitly) to enable OpenAI-based review.

If no API key is available, the reviewer agent falls back to a heuristic summary.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve repository root so that `agents` package is importable
repo_root = Path.cwd().resolve()
if not (repo_root / 'agents').exists():
    for parent in [repo_root] + list(repo_root.parents):
        if (parent / 'agents').exists():
            repo_root = parent
            break
sys.path.insert(0, str(repo_root))

from model_zoo.utils.env import read_env_file

# Load environment variables (including OPENAI_API_KEY) from .env if available
env_path = Path(os.environ.get('MODEL_ZOO_ENV_FILE', repo_root / '.env'))
if env_path.exists():
    values = read_env_file(env_path)
    for key, value in values.items():
        os.environ.setdefault(key, value)
    print(f'Loaded environment variables from {env_path}')
else:
    print(f'Environment file not found at {env_path}.')

if not os.environ.get('OPENAI_API_KEY') or os.environ['OPENAI_API_KEY'] == 'YOUR_OPENAI_API_KEY_HERE':
    os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_API_KEY_HERE')
    print('Update OPENAI_API_KEY in the cell above or .env before enabling the reviewer agent.')



In [ ]:
import numpy as np
from agents import ModelingAgent, EvaluationAgent, OpenAIReviewerAgent


In [ ]:
rng = np.random.default_rng(17)
X = rng.normal(size=(200, 4))
true_coef = np.array([1.5, -2.0, 0.7, 0.5])
logits = X @ true_coef
probabilities = 1 / (1 + np.exp(-logits))
y = (probabilities > 0.5).astype(int)

data_sources = {
    'feature_matrix': X.tolist(),
    'labels': y.tolist(),
}


In [ ]:
modeling_agent = ModelingAgent(run_timeout=240)
evaluation_agent = EvaluationAgent(
    metric_thresholds={'accuracy': 0.7},
    required_artifacts=['probabilities']
)
try:
    reviewer_agent = OpenAIReviewerAgent()
    print('OpenAI reviewer enabled.')
except Exception as exc:
    print(f'Reviewer unavailable: {exc}')
    reviewer_agent = OpenAIReviewerAgent(enabled=False)


In [ ]:
try:
    result = modeling_agent.execute_workflow(
        data_sources=data_sources,
        env='prod',
        # params can be overridden if needed

        context={'timeout_s': 180},
        evaluation_agent=evaluation_agent,
        reviewer_agent=reviewer_agent if reviewer_agent.enabled else None,
        register=False,
        model_id='logit_marketing_v1',
    )
    result
except Exception as exc:
    print(f'Workflow failed: {exc}')

